### Imports

In [13]:
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval import evaluate, metrics
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase
import pandas as pd
from deepeval.models import OllamaModel
from pathlib import Path
from deepeval.evaluate import AsyncConfig


# Walk up from notebook dir until we find the project root containing .env.local
_dir = Path.cwd()
while not (_dir / ".env.local").exists() and _dir != _dir.parent:
    _dir = _dir.parent
env_path = _dir / ".env.local"
print("Using:", env_path, "exists:", env_path.exists())

load_dotenv(env_path, override=True)

CLOUD_MODEL_BASE_URL = os.getenv("CLOUD_MODEL_BASE_URL")
LOCAL_MODEL_BASE_URL = os.getenv("LOCAL_MODEL_BASE_URL")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")


Using: /Users/michelecandolfo/Documents/workspaces/DeepEval/ai-engineering-portfolio/.env.local exists: True


### Initialise the Judge

In [2]:
judgeModel = OllamaModel(
    model="qwen3.5:cloud",
    base_url=CLOUD_MODEL_BASE_URL,
    temperature=0.0,
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},   
)

### Initialise the Candidate

In [3]:
candidateModel = ChatOllama(
    base_url=CLOUD_MODEL_BASE_URL,
    model="gpt-oss:20b-cloud",
    temperature=0.3,
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},  
)

###  Creating Test Data for Test Cases/Goldens

In [4]:
test_data = [
  {
    "input": (
      "Summarize the following text in 2-3 sentences:\n\n"
      "Integration testing verifies that modules and services work correctly when combined. "
      "It focuses on interfaces, data contracts, and error handling across boundaries. "
      "Effective integration tests run in realistic environments (e.g., staging with real databases or well-behaved test doubles) "
      "and help catch issues that unit tests miss, such as serialization errors, mismatched schemas, or race conditions. "
      "They typically come after unit tests and before system tests, providing confidence that the composition of parts behaves as intended."
    )
  },
  {
    "input": (
      "Summarize the following text in 2-3 sentences:\n\n"
      "Regression testing is executed after changes—like bug fixes, refactors, or dependency upgrades—to ensure existing behavior remains intact. "
      "Teams usually automate a critical subset to balance coverage and runtime, prioritizing high-risk flows and past incident areas. "
      "A reliable regression suite reduces release anxiety, shortens feedback cycles, and prevents the reintroduction of previously resolved defects."
    )
  },
  {
    "input": (
      "Summarize the following text in 2-3 sentences:\n\n"
      "Exploratory testing combines learning, test design, and execution in one activity. "
      "Testers use charters and heuristics to investigate software behavior, adapting based on findings rather than following rigid scripts. "
      "This approach often uncovers edge cases, usability problems, and gaps in requirements that scripted testing may overlook."
    )
  },
  {
    "input": (
      "Summarize the following text in 2-3 sentences:\n\n"
      "Performance testing evaluates how a system behaves under expected and peak loads. "
      "Key subtypes include load testing for steady traffic, stress testing to find breaking points, and soak testing for long-duration stability. "
      "Meaningful results rely on realistic data, representative scenarios, and precise telemetry to locate bottlenecks in CPU, memory, I/O, or external services."
    )
  },
  {
    "input": (
      "Summarize the following text in 2-3 sentences:\n\n"
      "Code reviews improve quality by catching defects early, sharing knowledge, and aligning on style and architecture. "
      "Effective reviews focus on correctness, readability, and testability rather than personal preference. "
      "Small, frequent pull requests with clear context, tests, and checklists reduce cognitive load and speed up approvals."
    )
  },
  {
    "input": (
      "Summarize the following text in 2-3 sentences:\n\n"
      "Continuous Integration and Continuous Delivery (CI/CD) automate build, test, and deployment steps to accelerate delivery while maintaining quality. "
      "Pipelines should be deterministic, fast, and observable, with guardrails like unit, integration, and end-to-end tests. "
      "Feature flags and progressive delivery techniques (e.g., canary releases) limit blast radius and enable rapid rollback."
    )
  },
  {
    "input": (
      "Summarize the following text in 2-3 sentences:\n\n"
      "Test doubles—mocks, stubs, fakes, and spies—isolate units by replacing collaborators with controlled stand-ins. "
      "Overusing mocks can make tests brittle and tied to implementation details, so prefer fakes or contract tests at boundaries. "
      "Clear seams and dependency injection improve testability and reduce setup complexity."
    )
  },
  {
    "input": (
      "Summarize the following text in 2-3 sentences:\n\n"
      "Flaky tests pass or fail nondeterministically due to timing issues, network variability, shared state, or environment drift. "
      "Mitigations include explicit waits with timeouts, isolating state, using idempotent test data, and stabilizing external dependencies with hermetic services. "
      "Tracking flake rates and quarantining unstable tests prevents signal erosion in CI."
    )
  }
]

In [5]:
goldens = [Golden(input=d["input"]) for d in test_data]
dataset = EvaluationDataset(goldens=goldens)

### Optional: Push the Goldens Data Set to Confident AI (for reuse with different LLMs)

In [6]:
dataset.push("Summarization Dataset")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=972721;https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/datasets/cmpcbt7kf000ak413lfsinyfl\https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/datasets/cmpcbt7kf000ak413lfsinyfl]8;;\

### Optional: Pull the current dataset and convert it into LLMTestCases
##### This is only needed if you already have a dataset in Confident AI and want to evalute different LLMs with it

In [57]:
#dataset.pull(alias="Summarization Dataset", auto_convert_goldens_to_test_cases=True) 

### Add actual output from the candidate LLM to the Goldens Data Set

In [7]:
for g in dataset.goldens:
    response = candidateModel.invoke(g.input)
    g.actual_output = getattr(response, "content", str(response))


#### Convert Goldens to LLMTestCases for Evaluation via DeepEval

In [8]:
test_cases = [
    LLMTestCase(
        input=g.input,
        actual_output=getattr(g, "actual_output", None)
    )
    for g in dataset.goldens
]

### Define metric

##### The summarization metric breaks the score into alignment_score and coverage_score.

<img src="images/Summarization.png" alt="Summarization" width="800">

##### The final score is the minumum of:

- alignment_score which determines whether the summary contains hallucinated or contradictory information to the original text.
- coverage_score which determines whether the summary contains the necessary information from the original text.


#### How to interpret the Summarization Score

| **Score Range** | **Meaning** | **Example Behavior** |
|------------------|-------------|-----------------------|
| **0.8 → 1.0** | 🟢 **Excellent summary** | Faithful to the source, includes all key points, concise and accurate |
| **0.6 → 0.8** | 🟡 **Good but imperfect** | Generally accurate, but missing minor details or slightly redundant |
| **0.3 → 0.6** | 🟠 **Weak summary** | Misses several key points, too short/long, or partially inaccurate |
| **0.0 → 0.3** | 🔴 **Poor summary** | Hallucinated or incorrect info, unrelated to source text |

In [9]:
metric = SummarizationMetric(model=judgeModel)

### Execute evaluation

In [16]:
results = evaluate(test_cases=test_cases, metrics=[metric], async_config=AsyncConfig(run_async=True, max_concurrent=3))

✨ You're running DeepEval's latest Summarization Metric! (using qwen3.5:cloud (Ollama), strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 1.0, threshold: 0.5, strict: False, evaluation model: qwen3.5:cloud (Ollama), reason: The score is 1.00 because the summary exhibits perfect fidelity to the source material, accurately conveying all key points with exceptional clarity and conciseness., error: None)

For test case:

  - input: Summarize the following text in 2-3 sentences:

Exploratory testing combines learning, test design, and execution in one activity. Testers use charters and heuristics to investigate software behavior, adapting based on findings rather than following rigid scripts. This approach often uncovers edge cases, usability problems, and gaps in requirements that scripted testing may overlook.
  - actual output: Exploratory testing blends learning, test design, and execution into a single activity, guiding testers with charters and heuristics to probe software behavior. Rather than following rigid scripts, testers adapt their approach based on findings, allowin

⚠ WARNING: No hyperparameters logged.
» ]8;id=296526;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=767069;https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/test-runs/cmpccmny6000gs6130qxbdk3e/test-cases\https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/test-runs/cmpccmny6000gs6130qxbdk3e/test-cases]8;;\

#### Display results in a pandas dataframe

In [62]:
rows = []
for tr in results.test_results:
    for m in tr.metrics_data:
        rows.append({
            "test_case": tr.name,
            "input": tr.input,
            "expected_output": tr.expected_output,
            "actual_output": tr.actual_output,
            "metric": m.name,
            "score": m.score,
            "threshold": m.threshold,
            "success": m.success,
            "reason": m.reason,
            "evaluation_model": m.evaluation_model,
            "evaluation_cost": m.evaluation_cost,
        })

results_df = pd.DataFrame(rows)
display(results_df)

,test_case,input,expected_output,actual_output,metric,score,threshold,success,reason,evaluation_model,evaluation_cost
0,test_case_0,Summarize the following text in 2-3 sentences:...,None,Integration testing ensures that modules and s...,Summarization,1.000000,0.5,True,The score is 1.00 because the summary accurate...,qwen3:latest (Ollama),0.0
1,test_case_2,Summarize the following text in 2-3 sentences:...,None,"Exploratory testing blends learning, test desi...",Summarization,1.000000,0.5,True,The score is 1.00 because the summary accurate...,qwen3:latest (Ollama),0.0
2,test_case_1,Summarize the following text in 2-3 sentences:...,None,Regression testing runs after changes—such as ...,Summarization,1.000000,0.5,True,The score is 1.00 because the summary accurate...,qwen3:latest (Ollama),0.0
3,test_case_3,Summarize the following text in 2-3 sentences:...,None,Performance testing evaluates how a system beh...,Summarization,1.000000,0.5,True,The score is 1.00 because the summary accurate...,qwen3:latest (Ollama),0.0
4,test_case_4,Summarize the following text in 2-3 sentences:...,None,Code reviews boost quality by catching defects...,Summarization,0.666667,0.5,True,The score is 0.67 because the summary includes...,qwen3:latest (Ollama),0.0
5,test_case_7,Summarize the following text in 2-3 sentences:...,None,Flaky tests nondeterministically pass or fail ...,Summarization,1.000000,0.5,True,The score is 1.00 because the summary accurate...,qwen3:latest (Ollama),0.0
6,test_case_6,Summarize the following text in 2-3 sentences:...,None,"Test doubles—including mocks, stubs, fakes, an...",Summarization,1.000000,0.5,True,The score is 1.00 because the summary accurate...,qwen3:latest (Ollama),0.0
7,test_case_5,Summarize the following text in 2-3 sentences:...,None,"CI/CD automates the build, test, and deploymen...",Summarization,0.666667,0.5,True,The score is 0.67 because the summary includes...,qwen3:latest (Ollama),0.0


#### Evaluate the results and add a suggestion for improvements

In [64]:
with pd.option_context("display.max_colwidth", None):
    failing = results_df[results_df["success"].astype(str).str.lower().eq("false")]
    display(failing)

,test_case,input,expected_output,actual_output,metric,score,threshold,success,reason,evaluation_model,evaluation_cost
